In [5]:
# A1 · Part 1 — Choose the right tool.  Starter notebook.
# PE6201 · Emerging AI Technologies. Run top to bottom, then replace the placeholder domain.

# A1 · Part 1 — Choose the right tool  *(starter)*

**What Part 1 asks of you.** Take a classification task in **your** domain — sorting things
into two or more categories. Build it **twice**: once as a **scikit-learn classifier** trained on
labelled examples, once as a **foundation-model API call** with a prompt and no training at all.
Measure both on **accuracy, cost, latency, and effort**. Then answer the question that matters:
*which would you actually deploy — and at what scale would your answer change?*

**What this notebook is.** The whole comparison, already working, on a **placeholder domain**
(houseplant care messages). Everything you must change is marked
**`### REPLACE ME`**. Nothing else needs editing.

**The placeholder domain is not a suggestion.** Do not submit houseplants. Replace it with the
domain you know — see the brief on picking one.

**Cost.** Roughly 60 short API calls ≈ well under one US cent on the default model. We count the
tokens at the end anyway, because *cost* is one of the four things you are measuring.

**Deliverable from this notebook:** this notebook (run, with output visible), a comparison table,
and roughly 400 words.

> **If a cell does not work.** This notebook was written before you ran it, against library
> versions, model names and provider behaviour that change without notice — so treat it as a
> working draft, not gospel. **Read the error, then try to fix it yourself:** debugging is part of
> the work, not a detour from it. Then **report it** — in class or by email — with the cell, the
> full error text, and what you tried, so the shared copy gets fixed and nobody behind you loses
> the same hour. Reporting a problem costs you nothing. The one thing not to do is quietly work
> around a broken cell: if your fix changes what the notebook measures, say so in your writeup.

### Setup

In [6]:
%pip -q install openai scikit-learn

Note: you may need to restart the kernel to use updated packages.


### Connect to OpenRouter
Same route as the Class 1, 2 and 3 notebooks. Your key was issued to you individually.
**Never type a key into a code cell** — this reads it from a Colab secret, an environment
variable, or a hidden prompt.

In [8]:
import os, getpass, time, json
from openai import OpenAI

def _get_openrouter_key():
    k = os.environ.get('MY_PRIVATE_OPENROUTER_KEY') or os.environ.get('OPENROUTER_API_KEY')
    if k: return k
    try:
        from google.colab import userdata
        k = userdata.get('OPENROUTER_API_KEY')
        if k: return k
    except Exception:
        pass
    return getpass.getpass('Paste your OpenRouter API key (hidden): ')

API_KEY = _get_openrouter_key()
client  = OpenAI(base_url='https://openrouter.ai/api/v1', api_key=API_KEY)

# TRY THIS (knob): any model your key can reach. Cheaper models are a fair comparison too —
# part of your argument may be that a small model is good enough.
#   'openai/gpt-4o-mini' (default)  ·  'anthropic/claude-3.5-haiku'
#   'qwen/qwen-2.5-7b-instruct'     ·  'deepseek/deepseek-chat'
GEN_MODEL = 'openai/gpt-4o-mini'
TOKENS = {'in': 0, 'out': 0}

def generate(prompt: str, system: str = None, max_new_tokens: int = 40,
             temperature: float = 0.0) -> str:
    """temperature 0 = deterministic. Essential when comparing two approaches:
       otherwise you cannot tell a real difference from random variation."""
    msgs = ([{'role': 'system', 'content': system}] if system else []) + \
           [{'role': 'user', 'content': prompt}]
    r = client.chat.completions.create(model=GEN_MODEL, messages=msgs,
                                       temperature=temperature, max_tokens=max_new_tokens)
    if getattr(r, 'usage', None):
        TOKENS['in']  += r.usage.prompt_tokens
        TOKENS['out'] += r.usage.completion_tokens
    return r.choices[0].message.content.strip()

print('connected · model:', GEN_MODEL, '·', generate('Reply with exactly the word: ready'))

connected · model: openai/gpt-4o-mini · ready


### Check your remaining credit  —  free, this call is not billed
Your allocation is **fixed and will not be topped up**, so watch it rather than discovering it at
the end. `usage` is what you have spent; `limit` is your cap; the difference is what is left.
Run this at the start of a working session and again at the end.

In [9]:
import urllib.request, json as _json

def credit_check():
    """Ask OpenRouter what is left on your key. Costs nothing."""
    try:
        req = urllib.request.Request('https://openrouter.ai/api/v1/key',
                                     headers={'Authorization': f'Bearer {API_KEY}'})
        d = _json.load(urllib.request.urlopen(req, timeout=20))['data']
        used, cap = d.get('usage'), d.get('limit')
        if cap is None:
            print(f'spent so far: ${used:.4f}   (no cap set on this key)')
        else:
            print(f'spent ${used:.4f} of ${cap:.2f}  ->  ${cap - used:.4f} remaining')
        rl = d.get('rate_limit') or {}
        if rl:
            print(f"rate limit: {rl.get('requests')} requests per {rl.get('interval')}")
    except Exception as e:
        print('could not check credit (not a problem, carry on):', str(e)[:120])

credit_check()

spent $0.4898 of $10.00  ->  $9.5102 remaining
rate limit: -1 requests per 10s


---
## 0 · Six words you need — *start here if you don't code*

You do not have to write much Python to finish this. Read the plain-English parts, swap the data,
run the cells, and watch four numbers.

| Word | In plain words |
|---|---|
| **Classification** | Sorting each item into one of a fixed set of categories. "Which of these three buckets does this belong in?" |
| **Label** | The correct category for one example. Supplied by a human — you. |
| **Training** | Showing a program many labelled examples until it can guess the label for a new one. |
| **Accuracy** | Of the items we tested, the fraction the system got right. One number, 0 to 1. |
| **Latency** | How long one prediction takes. Measured here in milliseconds per item. |
| **Baseline** | The score you would get by doing something stupid but cheap (e.g. always guessing the most common label). A result that cannot beat its baseline is not a result. |

**The point of Part 1 is not that one approach wins.** It is that you can say *which*, *on what
evidence*, and *at what scale your answer flips*.

---
## 1 · Your data  ### REPLACE ME

You need labelled examples: a short piece of text, and its correct category.

**How many?** 60–120 is plenty. Fewer than ~40 and the accuracy figures are too noisy to compare.

**How many categories?** Two or three. More than four and you will spend all your time labelling.

**Where from?** Write them yourself from memory of your domain, export them from something you
already have, or collect them by hand. **Do not have an AI assistant invent your dataset** — the
point is that you can tell a right answer from a wrong one, which requires the data to be real to
you. (If you must generate examples, say so in your writeup and explain what that costs you.)

**Balance them.** If 90% of your examples are one category, a system that always guesses that
category scores 90% while knowing nothing. Keep the categories roughly even, and always compare
against the majority-class baseline printed below.

In [10]:
### REPLACE ME — your categories. Two or three, named the way your domain names them.
LABELS = ['compliant','discrepant']

### REPLACE ME — your labelled examples: (text, label).
# The placeholder below is houseplant-care messages. Yours will be triage notes, support tickets,
# match reports, recipe steps, lyrics, job ads — whatever your domain actually produces.
DATA = [
    # ===== 金额类 =====
    ("invoice amount USD 34,200 against LC amount USD 31,500, tolerance plus minus 5 percent", 'discrepant'),
    ("invoice amount USD 32,800 within the plus minus 5 percent tolerance of LC amount USD 31,500", 'compliant'),
    ("cumulative drawings across two partial shipments total USD 145,000, exceeding LC amount USD 132,000 with no tolerance clause stated", 'discrepant'),
    ("invoice amount USD 58,900 below LC strict amount USD 61,200 with no tolerance clause stated", 'discrepant'),
    ("invoice amount USD 96,000 exactly matches LC amount USD 96,000, no tolerance needed", 'compliant'),
    ("invoice amount USD 41,300 within the plus minus 10 percent tolerance of LC amount USD 38,000", 'compliant'),
    ("invoice amount USD 44,500 against LC amount USD 38,000, tolerance plus minus 10 percent", 'discrepant'),
    ("invoice amount USD 178,000 against LC amount USD 165,000, tolerance plus minus 3 percent", 'discrepant'),
    ("invoice amount USD 168,400 within the plus minus 3 percent tolerance of LC amount USD 165,000", 'compliant'),
    ("cumulative drawings total USD 199,500 against LC amount USD 200,000, within LC limit", 'compliant'),

    # ===== 数量/描述类 =====
    ("goods described as stainless steel valve fittings on invoice but LC specifies brass valve fittings", 'discrepant'),
    ("quantity 8,600 units on packing list exceeds the plus minus 5 percent tolerance of LC quantity 8,000 units", 'discrepant'),
    ("quantity 21,300 cartons within tolerance of LC stated quantity 20,000 cartons, tolerance 10 percent", 'compliant'),
    ("invoice shows model number PZ-42 but LC requires model PZ-40", 'discrepant'),
    ("goods description reads 220gsm cotton fabric on invoice, matching LC wording exactly", 'compliant'),
    ("quantity 5,200 sets on invoice exceeds the plus minus 5 percent tolerance of LC quantity 5,000 sets", 'discrepant'),
    ("goods described as frozen shrimp grade A on invoice but LC specifies frozen shrimp grade AA", 'discrepant'),
    ("quantity 47.5 metric tons within the about 10 percent tolerance of LC quantity 45 metric tons", 'compliant'),
    ("invoice describes colour as charcoal grey but LC specifies colour as navy blue", 'discrepant'),
    ("quantity 12,050 units within the plus minus 3 percent tolerance of LC quantity 12,000 units", 'compliant'),

    # ===== 时间类 =====
    ("bill of lading on-board date 3 October 2026 is after LC latest shipment date 20 September 2026", 'discrepant'),
    ("documents presented on 12 October 2026, 22 days after shipment date 20 September 2026, exceeding the 21-day presentation period", 'discrepant'),
    ("shipment on 18 September 2026 is before LC latest shipment date 25 September 2026, documents presented within 15 days", 'compliant'),
    ("bill of lading dated 30 September 2026, after LC expiry date 28 September 2026", 'discrepant'),
    ("documents presented on 8 October 2026, within 15 days of shipment date 29 September 2026, before expiry 20 October 2026", 'compliant'),
    ("shipment on 5 November 2026 is after LC latest shipment date 30 October 2026", 'discrepant'),
    ("documents presented on 25 September 2026, 9 days after shipment date 16 September 2026, within the 10-day presentation period", 'compliant'),
    ("documents presented on 14 September 2026, 12 days after shipment date 2 September 2026, exceeding the 10-day presentation period", 'discrepant'),
    ("shipment on 1 August 2026 is before LC latest shipment date 15 August 2026, presentation within LC validity", 'compliant'),
    ("bill of lading dated 22 November 2026, after LC expiry date 15 November 2026", 'discrepant'),

    # ===== 单据缺失/份数类 =====
    ("LC requires a fumigation certificate but presentation is missing it", 'discrepant'),
    ("LC requires invoice in 3 originals but only 2 originals presented", 'discrepant'),
    ("required signature missing on the commercial invoice", 'discrepant'),
    ("all required documents present: commercial invoice, bill of lading, packing list, certificate of origin, matching LC document list", 'compliant'),
    ("LC requires an insurance certificate covering 110 percent of invoice value but presentation is missing it", 'discrepant'),
    ("all required documents present: invoice in 3 originals, packing list in 2 originals, phytosanitary certificate, matching LC requirements exactly", 'compliant'),
    ("LC requires a beneficiary's certificate confirming courier dispatch but presentation is missing it", 'discrepant'),
    ("packing list presented in 1 original but LC requires 2 originals", 'discrepant'),
    ("all required documents present: invoice, bill of lading, packing list, insurance certificate, all matching LC terms", 'compliant'),
    ("required countersignature missing on the inspection certificate", 'discrepant'),

    # ===== 提单/物流类 =====
    ("bill of lading consigned to order of applicant but LC requires consignment to order of issuing bank", 'discrepant'),
    ("LC prohibits transshipment but bill of lading shows transshipment at Colombo", 'discrepant'),
    ("bill of lading shows on-deck notation but LC requires clean on-board bill of lading", 'discrepant'),
    ("bill of lading consigned to order of issuing bank matching LC requirement, no transshipment, freight prepaid", 'compliant'),
    ("LC prohibits transshipment but bill of lading shows transshipment at Dubai", 'discrepant'),
    ("bill of lading blank endorsed and consigned to order of shipper matching LC requirement, no transshipment", 'compliant'),
    ("bill of lading marked freight collect but LC requires freight prepaid", 'discrepant'),
    ("bill of lading consigned to order of shipper matching LC requirement, one permitted transshipment at Singapore as allowed by LC", 'compliant'),
    ("bill of lading shows notation cargo shipped in damaged condition, not a clean bill of lading as required by LC", 'discrepant'),
    ("air waybill consigned to applicant directly but LC requires consignment to order of issuing bank", 'discrepant'),

    # ===== 签发机构类 =====
    ("certificate of origin issued by Local Trade Bureau instead of the LC-required Chamber of Commerce", 'discrepant'),
    ("inspection certificate issued by a local unlisted inspector, not an LC-approved body such as SGS or Lloyds", 'discrepant'),
    ("certificate of origin issued by CCPIT as required by the LC", 'compliant'),
    ("certificate of origin issued by Vietnam Chamber of Commerce as required by the LC", 'compliant'),
    ("inspection certificate issued by SGS as required by the LC", 'compliant'),
    ("certificate of origin issued by a freight forwarder instead of the LC-required Chamber of Commerce", 'discrepant'),
    ("inspection certificate issued by Bureau Veritas as required by the LC", 'compliant'),
    ("phytosanitary certificate issued by a private laboratory instead of the LC-required competent government authority", 'discrepant'),
    ("certificate of origin issued by Indian Chamber of Commerce as required by the LC", 'compliant'),
    ("inspection certificate issued by an unaccredited local firm, not an LC-approved body such as Lloyds", 'discrepant'),
]
# The placeholder set is small so it runs fast. YOURS SHOULD BE BIGGER — aim for 60–120.

texts  = [t for t, _ in DATA]
labels = [l for _, l in DATA]
from collections import Counter
counts = Counter(labels)
majority = counts.most_common(1)[0]
print(f'{len(DATA)} examples · {dict(counts)}')
print(f'MAJORITY-CLASS BASELINE: always guess "{majority[0]}" -> '
      f'{majority[1]/len(DATA):.1%} accuracy. Both approaches must beat this to mean anything.')

60 examples · {'discrepant': 36, 'compliant': 24}
MAJORITY-CLASS BASELINE: always guess "discrepant" -> 60.0% accuracy. Both approaches must beat this to mean anything.


### Split: train vs test
The classifier learns from the **training** half and is scored on the **test** half it has never
seen. The API call gets no training at all — but it is scored on **exactly the same test items**,
which is the only reason the two accuracy numbers can be compared.

In [11]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.35, random_state=0, stratify=labels)
print(f'{len(X_train)} training examples · {len(X_test)} test examples')

39 training examples · 21 test examples


---
## 2 · Approach A — a trained classifier

**TF-IDF** turns each message into a count of which words it contains, weighted so that common
words ("the", "it") count for little and distinctive words ("webbing", "soggy") count for a lot.
**Logistic regression** then learns which words push toward which category. No meaning, no
understanding — just word statistics. It is decades old, runs on a laptop, and costs nothing per
prediction.

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report

# TRY THIS (knob): ngram_range=(1,2) also counts word PAIRS ("soft mushy"). Often helps.
clf = make_pipeline(TfidfVectorizer(ngram_range=(1, 1)), LogisticRegression(max_iter=1000))

t0 = time.perf_counter()
clf.fit(X_train, y_train)
train_secs = time.perf_counter() - t0

t0 = time.perf_counter()
pred_clf = clf.predict(X_test)
clf_latency_ms = (time.perf_counter() - t0) / len(X_test) * 1000

acc_clf = accuracy_score(y_test, pred_clf)
print(f'trained in {train_secs:.2f}s · accuracy {acc_clf:.1%} · {clf_latency_ms:.2f} ms per item\n')
print(classification_report(y_test, pred_clf, zero_division=0))

trained in 0.01s · accuracy 81.0% · 0.07 ms per item

              precision    recall  f1-score   support

   compliant       1.00      0.50      0.67         8
  discrepant       0.76      1.00      0.87        13

    accuracy                           0.81        21
   macro avg       0.88      0.75      0.77        21
weighted avg       0.85      0.81      0.79        21



---
## 3 · Approach B — a foundation model, no training

No training data at all. We describe the job in words and hand the model one message at a time.

Two things to notice while you read the prompt. First, we **constrain the output** — "reply with
one word only, from this list" — because a model asked an open question will happily write a
paragraph, and a paragraph cannot be scored automatically. Second, the prompt below is
**deliberately plain**; improving it is one of the knobs at the end.

In [13]:
### REPLACE ME — describe YOUR task in one sentence. Keep the output constraint exactly as it is.
SYSTEM = (
   "You classify Letter of Credit (LC) document examination cases as compliant or discrepant "
    "based on whether the presented document details comply with the LC terms described in the message.\n"
    f"Reply with exactly one word from this list: {', '.join(LABELS)}.\n"
    "No explanation, no punctuation, no other words."
)

def classify_by_api(text: str) -> str:
    out = generate(f'Message: "{text}"\nCategory:', system=SYSTEM, max_new_tokens=6)
    out = out.strip().strip('.').lower()
    return out if out in LABELS else 'UNPARSEABLE'   # count failures, never hide them

t0 = time.perf_counter()
pred_api = [classify_by_api(t) for t in X_test]
api_latency_ms = (time.perf_counter() - t0) / len(X_test) * 1000

acc_api = accuracy_score(y_test, pred_api)
bad = sum(1 for p in pred_api if p == 'UNPARSEABLE')
print(f'accuracy {acc_api:.1%} · {api_latency_ms:.0f} ms per item · {bad} unparseable replies\n')
print(classification_report(y_test, pred_api, zero_division=0))

accuracy 100.0% · 1246 ms per item · 0 unparseable replies

              precision    recall  f1-score   support

   compliant       1.00      1.00      1.00         8
  discrepant       1.00      1.00      1.00        13

    accuracy                           1.00        21
   macro avg       1.00      1.00      1.00        21
weighted avg       1.00      1.00      1.00        21



### Look at the disagreements
Where the two approaches differ is where your writeup lives. Read these before you write a word.

In [14]:
print(f'{"TRUE":<12}{"CLASSIFIER":<12}{"API":<12}TEXT')
for txt, truth, a, b in zip(X_test, y_test, pred_clf, pred_api):
    if a != b:
        print(f'{truth:<12}{a:<12}{b:<12}{txt[:60]}')

TRUE        CLASSIFIER  API         TEXT
compliant   discrepant  compliant   shipment on 1 August 2026 is before LC latest shipment date 
compliant   discrepant  compliant   shipment on 18 September 2026 is before LC latest shipment d
compliant   discrepant  compliant   invoice amount USD 168,400 within the plus minus 3 percent t
compliant   discrepant  compliant   inspection certificate issued by SGS as required by the LC


---
## 4 · The four measurements

**Cost** is where most submissions go vague. The classifier costs nothing per prediction but costs
*you* the labelling time. The API costs a fraction of a cent per prediction and costs you almost
no setup. Which is cheaper depends entirely on **how many items you will ever classify** — which
is exactly the scale question the brief asks you.

In [15]:
### REPLACE ME — check today's price for YOUR model on openrouter.ai/models. These change.
PRICE_IN_PER_M  = 0.15    # US$ per 1,000,000 input tokens
PRICE_OUT_PER_M = 0.60    # US$ per 1,000,000 output tokens

api_cost = TOKENS['in']/1e6*PRICE_IN_PER_M + TOKENS['out']/1e6*PRICE_OUT_PER_M
per_item = api_cost / max(1, len(X_test))

### REPLACE ME — your honest effort estimate, in hours. Be truthful; it is part of the argument.
EFFORT = {'classifier': 'labelling 60 examples ≈ 1 h, then ~0 to run',
          'api':        'writing and testing the prompt ≈ 0.3 h, no labelling'}

print(f'{"":<14}{"ACCURACY":>10}{"COST / ITEM":>16}{"LATENCY":>12}   EFFORT')
print(f'{"baseline":<14}{majority[1]/len(DATA):>9.1%}{"$0":>16}{"~0 ms":>12}   none')
print(f'{"classifier":<14}{acc_clf:>9.1%}{"$0.00 (local)":>16}{clf_latency_ms:>9.2f} ms   {EFFORT["classifier"]}')
print(f'{"API":<14}{acc_api:>9.1%}{"$"+format(per_item,".6f"):>16}{api_latency_ms:>9.0f} ms   {EFFORT["api"]}')
print(f'\ntokens: {TOKENS["in"]} in / {TOKENS["out"]} out · this run cost about ${api_cost:.4f}')

for n in (1_000, 100_000, 10_000_000):
    print(f'  at {n:>10,} items/year -> API ≈ ${per_item*n:,.2f}   classifier ≈ $0 in API spend')

                ACCURACY     COST / ITEM     LATENCY   EFFORT
baseline          60.0%              $0       ~0 ms   none
classifier        81.0%   $0.00 (local)     0.07 ms   labelling 60 examples ≈ 1 h, then ~0 to run
API              100.0%       $0.000016     1246 ms   writing and testing the prompt ≈ 0.3 h, no labelling

tokens: 1971 in / 56 out · this run cost about $0.0003
  at      1,000 items/year -> API ≈ $0.02   classifier ≈ $0 in API spend
  at    100,000 items/year -> API ≈ $1.57   classifier ≈ $0 in API spend
  at 10,000,000 items/year -> API ≈ $156.79   classifier ≈ $0 in API spend


**Read that last block carefully.** At a thousand items a year the API is free in practice and the
labelling effort is the only real cost. At ten million, the arithmetic reverses and someone starts
asking why you are renting a frontier model to do a job TF-IDF does. *Where is your crossover, and
what would have to be true for you to change your mind?* That is your 400 words.

---
## 5 · YOUR TURN

1. Replace `LABELS`, `DATA`, and `SYSTEM` with your domain. Re-run from the top.
2. Check both approaches beat the majority-class baseline. If neither does, your categories are
   probably overlapping — fix the categories, not the code.
3. Read the disagreements. Pick **two** and work out *why* each approach got it wrong.
4. Fill in `PRICE_*` from today's OpenRouter pricing and `EFFORT` from your own honest reckoning.
5. Write the comparison table and ~400 words.

**Knobs worth trying** (each one is a sentence in your writeup if you report what moved):

- `ngram_range=(1,2)` on the vectoriser.
- Halve the training set: `test_size=0.65`. How much accuracy does the classifier lose? The API
  loses none — it never trained. **That gap is the real argument for the API.**
- Add two worked examples to `SYSTEM` (few-shot, Class 3 rung 2) and re-measure. Note that this
  raises your input tokens on *every* call, forever.
- Swap `GEN_MODEL` for a cheaper model. If accuracy holds, that is a finding.

### What your ~400 words must contain

- The comparison table: accuracy, cost, latency, effort, for both.
- Which you would deploy, and **why** — naming the trade-off you accepted.
- **The scale at which your answer changes**, with the arithmetic.
- One thing you could not measure here, and what it would take to measure it.

In [ ]:
# Scratch space for your own experiments.